# Multimodal Topic Mining and Image Classification
## Advanced Customer Analytics - Visual Data Predictions Assignment


In [ ]:
# Install required packages
# !pip install -q bertopic
# !pip install -q scikit-learn
# !pip install -q pillow
# !pip install -q pandas
# !pip install -q matplotlib
# !pip install -q seaborn
# !pip install -q requests
# !pip install -q tqdm
# !pip install -q datasets
# !pip install -q kaggle


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import zipfile
import subprocess


In [90]:
def download_and_process_pokemon_cards(save_dir='pokemon_cards', num_cards=1000):
    """
    Download Pokemon TCG dataset from Kaggle and download card images.
    Everything is saved in a single directory.
    
    Args:
        save_dir: Directory to save everything (images and metadata)
        num_cards: Number of cards to process
        
    Returns:
        df_final: Processed dataframe with image paths
    """
    
    # Create directory structure
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    temp_dir = f"{save_dir}/temp"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Step 1: Download dataset from Kaggle
    print("Downloading Pokemon TCG dataset from Kaggle...")
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'adampq/pokemon-tcg-all-cards-1999-2023', '-p', temp_dir],
        capture_output=True,
        text=True
    )
    
    # Step 2: Extract the dataset
    zip_path = f'{temp_dir}/pokemon-tcg-all-cards-1999-2023.zip'
    if os.path.exists(zip_path):
        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
    
    # Step 3: Load the CSV
    csv_files = [f for f in os.listdir(temp_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in dataset directory")
    
    cards_file = f'{temp_dir}/{csv_files[0]}'
    df_raw = pd.read_csv(cards_file)
    print(f"Loaded {len(df_raw)} total cards from Kaggle dataset\n")
    
    # Step 4: Download images for subset
    df = df_raw.head(num_cards).copy()
    print(f"Processing {len(df)} cards...")
    
    successful = []
    failed = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading images"):
        try:
            card_id = row['id']
            set_code, number = card_id.split('-')
            
            downloaded = False
            for url_template in [
                f"https://images.pokemontcg.io/{set_code}/{number}_hires.png",
                f"https://images.pokemontcg.io/{set_code}/{number}.png"
            ]:
                try:
                    response = requests.get(url_template, timeout=10)
                    response.raise_for_status()
                    
                    img = Image.open(BytesIO(response.content)).convert('RGB')
                    img_path = f"{save_dir}/images/card_{idx:05d}.jpg"
                    img.save(img_path, 'JPEG', quality=95)
                    
                    df.at[idx, 'image_path'] = img_path
                    successful.append(idx)
                    downloaded = True
                    break
                except:
                    continue
            
            if not downloaded:
                failed.append(card_id)
                
        except Exception as e:
            failed.append(row.get('id', f'idx_{idx}'))
            continue
    
    # Step 5: Save only the final processed dataset
    df_final = df.loc[successful].copy()
    csv_path = f'{save_dir}/pokemon_cards_metadata.csv'
    df_final.to_csv(csv_path, index=False)
    
    # Step 6: Clean up temporary files
    print("\nCleaning up temporary files...")
    import shutil
    shutil.rmtree(temp_dir)
    
    print(f"Successfully processed: {len(df_final)} cards")
    print(f"Failed to download: {len(failed)} cards")
    print(f"Images saved to: {save_dir}/images/")
    print(f"Metadata saved to: {csv_path}")
    
    return df_final

In [ ]:
save_dir = 'pokemon_cards'

# Download and process the dataset
df = download_and_process_pokemon_cards(save_dir, num_cards=6000)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df_radfw.columns)}):")
for col in df.columns:
    print(f"  - {col}")

print(f"\nFirst few rows:")
df.head()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Extracting dataset...
Loaded 17172 total cards from Kaggle dataset

Processing 6000 cards...


### Load the dataset and preprocess it

In [21]:
# Load the processed data
cards_df = pd.read_csv('./pokemon_cards/pokemon_cards_metadata.csv')

print(f"Dataset shape: {cards_df.shape}")
print(f"\nColumns: {list(cards_df.columns)}")
cards_df.head()

Dataset shape: (5998, 30)

Columns: ['id', 'set', 'series', 'publisher', 'generation', 'release_date', 'artist', 'name', 'set_num', 'types', 'supertype', 'subtypes', 'level', 'hp', 'evolvesFrom', 'evolvesTo', 'abilities', 'attacks', 'weaknesses', 'retreatCost', 'convertedRetreatCost', 'rarity', 'flavorText', 'nationalPokedexNumbers', 'legalities', 'resistances', 'rules', 'regulationMark', 'ancientTrait', 'image_path']


,id,set,series,publisher,generation,release_date,artist,name,set_num,types,...,convertedRetreatCost,rarity,flavorText,nationalPokedexNumbers,legalities,resistances,rules,regulationMark,ancientTrait,image_path
0,base1-1,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Alakazam,1,['Psychic'],...,3.0,Rare Holo,Its brain can outperform a supercomputer. Its ...,[65],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN,pokemon_cards/images/card_00000.jpg
1,base1-2,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Blastoise,2,['Water'],...,3.0,Rare Holo,A brutal Pokémon with pressurized water jets o...,[9],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN,pokemon_cards/images/card_00001.jpg
2,base1-3,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Chansey,3,['Colorless'],...,1.0,Rare Holo,A rare and elusive Pokémon that is said to bri...,[113],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00002.jpg
3,base1-4,Base,Base,WOTC,First,1/9/1999,Mitsuhiro Arita,Charizard,4,['Fire'],...,3.0,Rare Holo,Spits fire that is hot enough to melt boulders...,[6],{'unlimited': 'Legal'},"[{'type': 'Fighting', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00003.jpg
4,base1-5,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Clefairy,5,['Colorless'],...,1.0,Rare Holo,Its magical and cute appeal has many admirers....,[35],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00004.jpg


In [22]:
import ast

def safe_parse_list(value):
    """Convert string representation of list to actual list."""
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return None
    return value

# Parse list columns
list_columns = ['types', 'subtypes', 'retreatCost', 'nationalPokedexNumbers']
for col in list_columns:
    cards_df[f'{col}_parsed'] = cards_df[col].apply(safe_parse_list)

# Extract first type (most important)
cards_df['primary_type'] = cards_df['types_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

# Extract first subtype
cards_df['primary_subtype'] = cards_df['subtypes_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

# Convert retreat cost to numeric
cards_df['retreat_cost_count'] = cards_df['retreatCost_parsed'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print("Parsed list columns")
cards_df[['name', 'primary_type', 'primary_subtype', 'hp', 'rarity']].head(10)

Parsed list columns


,name,primary_type,primary_subtype,hp,rarity
0,Alakazam,Psychic,Stage 2,80.0,Rare Holo
1,Blastoise,Water,Stage 2,100.0,Rare Holo
2,Chansey,Colorless,Basic,120.0,Rare Holo
3,Charizard,Fire,Stage 2,120.0,Rare Holo
4,Clefairy,Colorless,Basic,40.0,Rare Holo
5,Gyarados,Water,Stage 1,100.0,Rare Holo
6,Hitmonchan,Fighting,Basic,70.0,Rare Holo
7,Machamp,Fighting,Stage 2,100.0,Rare Holo
8,Magneton,Lightning,Stage 1,60.0,Rare Holo
9,Mewtwo,Psychic,Basic,60.0,Rare Holo


In [23]:
# Convert HP to numeric (some might be strings)
cards_df['hp_numeric'] = pd.to_numeric(cards_df['hp'], errors='coerce')

# Count attacks
cards_df['num_attacks'] = cards_df['attacks'].apply(
    lambda x: len(safe_parse_list(x)) if pd.notna(x) else 0
)

# Boolean: has weakness/resistance
cards_df['has_weakness'] = cards_df['weaknesses'].notna()
cards_df['has_resistance'] = cards_df['resistances'].notna()

print("Created numeric features")
cards_df[['name', 'hp_numeric', 'num_attacks', 'retreat_cost_count', 'has_weakness']].head(10)

Created numeric features


,name,hp_numeric,num_attacks,retreat_cost_count,has_weakness
0,Alakazam,80.0,1,3,True
1,Blastoise,100.0,1,3,True
2,Chansey,120.0,2,1,True
3,Charizard,120.0,1,3,True
4,Clefairy,40.0,2,1,True
5,Gyarados,100.0,2,3,True
6,Hitmonchan,70.0,2,2,True
7,Machamp,100.0,1,3,True
8,Magneton,60.0,2,1,True
9,Mewtwo,60.0,2,3,True


In [24]:
# Keep only Pokemon cards (they have types and HP)
pokemon_only = cards_df[
    (cards_df['supertype'] == 'Pokémon') & 
    (cards_df['primary_type'].notna())
].copy()

print(f"Original: {len(cards_df)} cards")
print(f"Pokemon only: {len(pokemon_only)} cards")
print(f"\nType distribution:")
print(pokemon_only['primary_type'].value_counts())

Original: 5998 cards
Pokemon only: 5119 cards

Type distribution:
primary_type
Grass        849
Water        814
Colorless    795
Psychic      690
Fighting     577
Fire         490
Lightning    476
Darkness     218
Metal        201
Dragon         9
Name: count, dtype: int64


In [25]:
print(f"\nDataset size: {len(pokemon_only)} Pokemon cards")

print(f"\nMissing values in key columns:")
key_cols = ['primary_type', 'hp_numeric', 'rarity', 'artist', 'num_attacks']
for col in key_cols:
    missing = pokemon_only[col].isna().sum()
    pct = (missing / len(pokemon_only)) * 100
    print(f"  {col}: {missing} ({pct:.1f}%)")



Dataset size: 5119 Pokemon cards

Missing values in key columns:
  primary_type: 0 (0.0%)
  hp_numeric: 0 (0.0%)
  rarity: 73 (1.4%)
  artist: 0 (0.0%)
  num_attacks: 0 (0.0%)


## Assignment Function 1: TOPIC_MINER()

Multimodal topic mining using BERTopic with CLIP embeddings.

**Implementation:** Combines text captions and images to discover topics in Pokemon cards.


In [26]:
def topic_miner(dataset_path, num_topics_target=None, show_evaluation=True, n_neighbors=25, 
    n_components=15, min_dist=0.0, umap_metric='cosine', min_cluster_size=40, min_samples=8, hdbscan_metric='euclidean', 
    cluster_selection_method='eom', cluster_selection_epsilon=0.15
):
    """    
    Reads dataset and uses BERTopic to identify multimodal topics.
    Labels each image with the topic assigned by BERTopic.

    Args:
        dataset_path: Path to CSV file with Pokemon card metadata
        num_topics_target: Target number of topics (None = automatic)
        min_cluster_size: HDBSCAN parameter for cluster granularity
        min_samples: HDBSCAN parameter for clustering sensitivity
        show_evaluation: Whether to display topic quality evaluation
        
    Returns:
        DataFrame with columns: [image_path, caption, topic_id, topic_label]
    """
    import pandas as pd
    import ast
    from PIL import Image
    from bertopic import BERTopic
    from bertopic.backend import MultiModalBackend
    from sentence_transformers import SentenceTransformer
    from hdbscan import HDBSCAN
    from umap import UMAP
    from sklearn.feature_extraction.text import CountVectorizer
    import numpy as np
    
    def _safe_parse_list(value):
        """Parse string representation of list to actual list."""
        if pd.isna(value):
            return None
        if isinstance(value, str):
            try:
                return ast.literal_eval(value)
            except:
                return None
        return value
    
    def _load_and_preprocess(dataset_path):
        """Load dataset and preprocess to Pokemon-only dataframe."""
        print("\n[1/6] Loading and preprocessing dataset...")
        df = pd.read_csv(dataset_path)
        print(f"   Loaded {len(df)} total cards")
        
        # Parse list columns
        list_columns = ['types', 'subtypes', 'retreatCost', 'nationalPokedexNumbers']
        for col in list_columns:
            df[f'{col}_parsed'] = df[col].apply(_safe_parse_list)
        
        # Extract features
        df['primary_type'] = df['types_parsed'].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
        )
        df['primary_subtype'] = df['subtypes_parsed'].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
        )
        df['retreat_cost_count'] = df['retreatCost_parsed'].apply(
            lambda x: len(x) if isinstance(x, list) else 0
        )
        df['hp_numeric'] = pd.to_numeric(df['hp'], errors='coerce')
        df['num_attacks'] = df['attacks'].apply(
            lambda x: len(_safe_parse_list(x)) if pd.notna(x) else 0
        )
        df['has_weakness'] = df['weaknesses'].notna()
        df['has_resistance'] = df['resistances'].notna()
        
        # Filter to Pokemon only
        pokemon_df = df[(df['supertype'] == 'Pokémon') & (df['primary_type'].notna())].copy()
        print(f"   Filtered to {len(pokemon_df)} Pokemon cards")
        
        return pokemon_df
    
    def _generate_captions(df):
        """Generate rich captions for all cards."""
        print("\n[2/6] Generating captions...")
        
        def create_caption(row):
            """Full metadata caption - realistic card description"""
            parts = []
            
            # Pokemon name (cleaned of prefixes)
            if pd.notna(row['name']):
                name = row['name']
                # Remove trainer prefixes
                for prefix in ["Blaine's ", "Brock's ", "Misty's ", "Sabrina's ", "Koga's ", 
                              "Erika's ", "Giovanni's ", "Lt. Surge's ", "Rocket's ", "Dark "]:
                    name = name.replace(prefix, "")
                parts.append(name)
            
            # Evolution info
            if pd.notna(row['primary_subtype']):
                parts.append(f"{row['primary_subtype']} stage")
            
            if pd.notna(row['evolvesFrom']):
                parts.append(f"Evolves from {row['evolvesFrom']}")
            
            # Stats
            if pd.notna(row['hp_numeric']):
                parts.append(f"HP {int(row['hp_numeric'])}")
            
            if row['num_attacks'] > 0:
                parts.append(f"{row['num_attacks']} attacks")
            
            if row['retreat_cost_count'] > 0:
                parts.append(f"Retreat cost {row['retreat_cost_count']}")
            
            # Battle characteristics
            if row['has_weakness']:
                parts.append("Has weakness")
            if row['has_resistance']:
                parts.append("Has resistance")
            
            # Metadata
            if pd.notna(row['generation']):
                gen_map = {'First': 'Gen 1', 'Second': 'Gen 2', 
                          'Third': 'Gen 3', 'Fourth': 'Gen 4',
                          'Fifth': 'Gen 5', 'Sixth': 'Gen 6',
                          'Seventh': 'Gen 7', 'Eighth': 'Gen 8'}
                parts.append(gen_map.get(row['generation'], row['generation']))
            
            if pd.notna(row['rarity']):
                parts.append(row['rarity'])
            
            # KEPT: Complete card metadata including name
            # Justification: Testing if multimodal fusion improves over text/image alone
            
            return ". ".join(parts) if parts else "Pokemon card"
        
        df['caption'] = df.apply(create_caption, axis=1)
        print(f"   Generated {len(df)} captions")
        return df
    
    def _load_images(df):
        """Load images and filter to valid ones."""
        print("\n[3/6] Loading images...")
        
        images = []
        valid_indices = []
        
        for idx, row in df.iterrows():
            try:
                img = Image.open(row['image_path']).convert('RGB')
                images.append(img)
                valid_indices.append(idx)
            except:
                continue
        
        df_with_images = df.loc[valid_indices].reset_index(drop=True)
        print(f"   Loaded {len(images)} images successfully")
        
        return images, df_with_images
    
    def _build_and_fit_model(descriptions, images, min_cluster_size, min_samples):
        """Build and fit BERTopic model."""
        print("\n[4/6] Building multimodal topic model...")
        
        multimodal_model = SentenceTransformer('clip-ViT-B-32')
        multimodal_backend = MultiModalBackend(embedding_model=multimodal_model)
        
        umap_model = UMAP(
            n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist,
            metric=umap_metric, random_state=42
        )
        
        hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size, min_samples=min_samples,
            metric=hdbscan_metric, cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon, prediction_data=True
        )
        
        vectorizer_model = CountVectorizer(
            stop_words='english', min_df=2, ngram_range=(1, 2)
        )
        
        topic_model = BERTopic(
            embedding_model=multimodal_backend, umap_model=umap_model,
            hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
            top_n_words=10, verbose=False
        )
        
        print("\n[5/6] Fitting BERTopic (this may take several minutes)...")
        topics, probs = topic_model.fit_transform(documents=descriptions, images=images)
        
        num_topics = len(set(topics)) - 1
        num_outliers = sum([1 for t in topics if t == -1])
        print(f"   ✓ Complete! Topics: {num_topics}, Outliers: {num_outliers}")
        
        return topics, topic_model
    
    def _create_labeled_dataset(df, topics):
        """Create final labeled dataset with topic IDs and labels."""
        print("\n[6/6] Creating labeled dataset...")
        
        df['topic_id'] = topics
        
        # Create topic labels
        topic_names = {}
        for topic_id in sorted([t for t in set(topics) if t != -1]):
            topic_cards = df[df['topic_id'] == topic_id]
            dominant_type = topic_cards['primary_type'].mode()[0] if len(topic_cards) > 0 else "Unknown"
            topic_names[topic_id] = f"Topic_{topic_id}_{dominant_type}"
        topic_names[-1] = "Topic_-1_Outlier"
        
        df['topic_label'] = df['topic_id'].map(topic_names)
        
        result_df = df[['image_path', 'caption', 'topic_id', 'topic_label', 
                        'primary_type', 'primary_subtype', 'hp_numeric', 'name']].copy()
        
        return result_df
    
    def _evaluate_topics(df, topics):
        """Display comprehensive topic quality evaluation."""
        print("\n" + "="*80)
        print("TOPIC MODELING RESULTS - QUALITY EVALUATION")
        print("="*80)
        
        num_topics = len(set(topics)) - 1
        num_outliers = sum([1 for t in topics if t == -1])
        outlier_rate = (num_outliers / len(topics)) * 100
        
        print(f"\nNumber of topics discovered: {num_topics}")
        print(f"Outliers: {num_outliers}/{len(topics)} ({outlier_rate:.1f}%)")
        print(f"Target: 12-18 topics, <10% outliers")
        
        # Topic purity
        print("\n" + "-" * 80)
        print("TOPIC PURITY (Does each topic = one Pokemon type?)")
        print("-" * 80)
        
        for topic_id in sorted([t for t in set(topics) if t != -1]):
            topic_cards = df[df['topic_id'] == topic_id]
            type_dist = topic_cards['primary_type'].value_counts(normalize=True) * 100
            
            dominant_type = type_dist.index[0]
            purity = type_dist.iloc[0]
            count = len(topic_cards)
            status = "GOOD" if purity >= 80 else "POOR"
            
            print(f"Topic {topic_id:2d}: {dominant_type:12s} - {count:3d} cards - {purity:5.1f}% pure - {status}")
            
            if purity < 80 and len(type_dist) > 1:
                contaminant = type_dist.index[1]
                contaminant_pct = type_dist.iloc[1]
                print(f"          WARNING: {contaminant_pct:.1f}% {contaminant} contamination")
        
        # Type coverage
        print("\n" + "-" * 80)
        print("TYPE COVERAGE (Does every type have a topic?)")
        print("-" * 80)
        
        all_types = df['primary_type'].value_counts()
        print(f"\nTotal types in dataset: {len(all_types)}")
        
        for ptype in all_types.index:
            type_cards = df[df['primary_type'] == ptype]
            non_outlier = type_cards[type_cards['topic_id'] != -1]
            
            if len(non_outlier) > 0:
                main_topic = non_outlier['topic_id'].mode()[0]
                in_topic = sum(non_outlier['topic_id'] == main_topic)
                coverage = (in_topic / len(type_cards)) * 100
                status = "GOOD" if coverage >= 70 else "POOR"
                print(f"{ptype:12s}: {len(type_cards):3d} cards -> Topic {main_topic:2d} ({coverage:5.1f}% coverage) - {status}")
            else:
                print(f"{ptype:12s}: {len(type_cards):3d} cards -> NO TOPIC (all outliers) - POOR")
        
        # Summary
        print("\n" + "-" * 80)
        print("SUMMARY")
        print("-" * 80)
        
        good_topics = sum([1 for t in set(topics) if t != -1 and 
                          (df[df['topic_id'] == t]['primary_type'].value_counts(normalize=True).iloc[0] * 100) >= 80])
        
        print(f"Clean topics (>80% purity): {good_topics}/{num_topics}")
        print(f"Outlier rate: {outlier_rate:.1f}% (target: <10%)")
        
        if good_topics >= num_topics * 0.8 and outlier_rate < 10:
            print("\nRESULT: EXCELLENT - Topics are clean and well-separated")
        elif good_topics >= num_topics * 0.6 and outlier_rate < 15:
            print("\nRESULT: GOOD - Most topics are clean, some mixing")
        else:
            print("\nRESULT: NEEDS IMPROVEMENT - Adjust clustering parameters")
        
        # Sample cards
        print("\n" + "-" * 80)
        print("TOPIC EXAMPLES (First 5 topics)")
        print("-" * 80)
        
        for topic_id in range(min(5, num_topics)):
            topic_cards = df[df['topic_id'] == topic_id]
            
            if len(topic_cards) > 0:
                dominant_type = topic_cards['primary_type'].mode()[0]
                type_pct = (topic_cards['primary_type'] == dominant_type).sum() / len(topic_cards) * 100
                
                print(f"\nTopic {topic_id}: {dominant_type} ({len(topic_cards)} cards, {type_pct:.0f}% purity)")
                print("  Sample cards:")
                for idx, row in topic_cards.head(3).iterrows():
                    print(f"    - {row['name']:20s} | Type: {row['primary_type']:10s} | {row['primary_subtype']:8s} | HP: {int(row['hp_numeric']):3d}")
        
        print("\n" + "="*80)
    
    pokemon_df = _load_and_preprocess(dataset_path)
    pokemon_df = _generate_captions(pokemon_df)
    images, pokemon_with_images = _load_images(pokemon_df)
    descriptions = pokemon_with_images['caption'].tolist()
    topics, topic_model = _build_and_fit_model(descriptions, images, min_cluster_size, min_samples)
    result_df = _create_labeled_dataset(pokemon_with_images, topics)
    
    if show_evaluation:
        _evaluate_topics(result_df, topics)
    
    print("\n" + "="*80)
    print(f"TOPIC_MINER() COMPLETE - Labeled {len(result_df)} images")
    print("="*80)
    
    return result_df

In [27]:
# Target: ~100 topics, ~8% outliers
balanced = topic_miner('./pokemon_cards/pokemon_cards_metadata.csv',
    n_neighbors=15,
    n_components=10,
    min_cluster_size=12,
    min_samples=3,
    cluster_selection_epsilon=0.05,  # light merging
    show_evaluation=True
)


[1/6] Loading and preprocessing dataset...
   Loaded 5998 total cards
   Filtered to 5119 Pokemon cards

[2/6] Generating captions...
   Generated 5119 captions

[3/6] Loading images...
   Loaded 5119 images successfully

[4/6] Building multimodal topic model...

[5/6] Fitting BERTopic (this may take several minutes)...
   ✓ Complete! Topics: 234, Outliers: 432

[6/6] Creating labeled dataset...

TOPIC MODELING RESULTS - QUALITY EVALUATION

Number of topics discovered: 234
Outliers: 432/5119 (8.4%)
Target: 12-18 topics, <10% outliers

--------------------------------------------------------------------------------
TOPIC PURITY (Does each topic = one Pokemon type?)
--------------------------------------------------------------------------------
Topic  0: Colorless    -  65 cards -  24.6% pure - POOR
Topic  1: Psychic      -  58 cards - 100.0% pure - GOOD
Topic  2: Fire         -  55 cards -  80.0% pure - GOOD
Topic  3: Lightning    -  55 cards -  92.7% pure - GOOD
Topic  4: Colorless  

In [28]:
# Quick comparison: Text vs Image vs Multimodal
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sentence_transformers import SentenceTransformer
from PIL import Image
import numpy as np

print("="*80)
print("QUICK TEST: Type Prediction (Text vs Image vs Multimodal)")
print("="*80)

# Prepare data
df_clean = balanced[balanced["topic_id"] != -1].copy()
train_df, test_df = train_test_split(df_clean, test_size=0.3, random_state=42, stratify=df_clean["primary_type"])

model = SentenceTransformer("clip-ViT-B-32")
results = {}

# Test each approach
for approach in ["text_only", "image_only", "multimodal"]:
    print(f"\nTesting {approach.upper()}...")
    
    if approach == "text_only":
        X_train = model.encode(train_df["caption"].tolist(), show_progress_bar=False)
        X_test = model.encode(test_df["caption"].tolist(), show_progress_bar=False)
    elif approach == "image_only":
        train_imgs = [Image.open(p).convert("RGB") for p in train_df["image_path"]]
        test_imgs = [Image.open(p).convert("RGB") for p in test_df["image_path"]]
        X_train = model.encode(train_imgs, show_progress_bar=True)
        X_test = model.encode(test_imgs, show_progress_bar=True)
    else:
        X_train_txt = model.encode(train_df["caption"].tolist(), show_progress_bar=False)
        X_test_txt = model.encode(test_df["caption"].tolist(), show_progress_bar=False)
        train_imgs = [Image.open(p).convert("RGB") for p in train_df["image_path"]]
        test_imgs = [Image.open(p).convert("RGB") for p in test_df["image_path"]]
        X_train_img = model.encode(train_imgs, show_progress_bar=True)
        X_test_img = model.encode(test_imgs, show_progress_bar=True)
        X_train = np.concatenate([X_train_txt, X_train_img], axis=1)
        X_test = np.concatenate([X_test_txt, X_test_img], axis=1)
    
    # Train and evaluate
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced", n_jobs=-1)
    clf.fit(X_train, train_df["primary_type"].values)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(test_df["primary_type"].values, y_pred)
    results[approach] = acc
    print(f"  Accuracy: {acc*100:.2f}%")

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"Text-only:   {results['text_only']*100:5.2f}%")
print(f"Image-only:  {results['image_only']*100:5.2f}%")
print(f"Multimodal:  {results['multimodal']*100:5.2f}%")
print("="*80)

if results["image_only"] > results["text_only"]:
    print("\n✓ GOOD: Images are more informative than text (as expected for visual task)")
else:
    print("\n⚠ WARNING: Text still dominates - may have data leakage")

if results["multimodal"] > max(results["text_only"], results["image_only"]):
    gain = (results["multimodal"] - max(results["text_only"], results["image_only"]))*100
    print(f"✓ GOOD: Multimodal fusion provides {gain:.2f}% improvement")
else:
    print("⚠ WARNING: Multimodal does not improve over best unimodal")

QUICK TEST: Type Prediction (Text vs Image vs Multimodal)

Testing TEXT_ONLY...
  Accuracy: 87.42%

Testing IMAGE_ONLY...


Batches: 100%|██████████| 44/44 [00:30<00:00,  1.45it/s]


  Accuracy: 91.47%

Testing MULTIMODAL...


Batches: 100%|██████████| 44/44 [00:30<00:00,  1.43it/s]
python(8626) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8627) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8628) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8629) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8630) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8631) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Accuracy: 94.10%

RESULTS SUMMARY
Text-only:   87.42%
Image-only:  91.47%
Multimodal:  94.10%

✓ GOOD: Images are more informative than text (as expected for visual task)
✓ GOOD: Multimodal fusion provides 2.63% improvement


## Assignment Function 2: PREDICT()

Train and test a classifier using BERTopic topic labels as classes.

**Implementation:** Compares Logistic Regression and Random Forest classifiers using **IMAGE-ONLY** embeddings to avoid data leakage (Pokemon names in captions directly reveal types).

**Key Changes:**
- ✅ Uses IMAGE-ONLY embeddings (not multimodal) to avoid 99%+ trivial accuracy
- ✅ Removed Gradient Boosting (too slow with minimal gain)
- ✅ Provides realistic visual classification challenge

In [ ]:
def predict(labeled_df, test_size=0.3, random_state=42, compare_classifiers=True):
    """
    Train and test classifiers using BERTopic topic labels as classes.
    
    NOTE: Uses IMAGE-ONLY embeddings to avoid data leakage. The captions contain
    Pokemon names which directly reveal their types, making prediction trivial (99%+ accuracy).
    Using only images provides a realistic visual classification challenge.
    
    Args:
        labeled_df: DataFrame output from topic_miner() with topic labels
        test_size: Proportion of data for testing (default: 0.3 = 30%)
        random_state: Random seed for reproducibility
        compare_classifiers: If True, compare Logistic Regression & Random Forest
        
    Returns:
        results_dict: Dictionary containing:
            - 'best_model': The trained classifier with best performance
            - 'best_classifier_name': Name of best classifier
            - 'confusion_matrix': Confusion matrix for best model
            - 'accuracy': Test accuracy for best model
            - 'classification_report': Detailed metrics per class
            - 'X_train': Training features (for further analysis)
            - 'X_test': Test features (for further analysis)
            - 'y_train': Training labels
            - 'y_test': Test labels
            - 'y_pred': Predictions on test set
            - 'comparison_results': Performance comparison of all classifiers (if compare_classifiers=True)
    """
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
    from sentence_transformers import SentenceTransformer
    from PIL import Image
    import numpy as np
    import pandas as pd
    
    print("="*80)
    print("PREDICT() - CLASSIFIER TRAINING AND EVALUATION")
    print("="*80)
    
    # Step 1: Filter out outliers (topic -1)
    print("\n[1/5] Preprocessing data...")
    df_filtered = labeled_df[labeled_df['topic_id'] != -1].copy()
    num_outliers = len(labeled_df) - len(df_filtered)
    print(f"   Removed {num_outliers} outliers (topic -1)")
    print(f"   Training with {len(df_filtered)} samples across {df_filtered['topic_id'].nunique()} topics")
    
    # Check class distribution
    topic_counts = df_filtered['topic_id'].value_counts()
    min_samples = topic_counts.min()
    if min_samples < 10:
        print(f"   ⚠ WARNING: Smallest class has only {min_samples} samples - may affect performance")
    
    # Step 2: Generate IMAGE-ONLY embeddings (to avoid data leakage)
    print("\n[2/5] Generating image embeddings (CLIP)...")
    print("   ⚠️  Using IMAGE-ONLY to avoid data leakage")
    print("   💡 Captions contain Pokemon names → types → trivial prediction (99%+ acc)")
    print("   💡 Image-only provides realistic visual classification challenge")
    model = SentenceTransformer('clip-ViT-B-32')
    
    # Image embeddings only
    print("\n   Encoding images...")
    images = [Image.open(path).convert('RGB') for path in df_filtered['image_path']]
    image_embeddings = model.encode(
        images,
        show_progress_bar=True,
        batch_size=32
    )
    
    print(f"   Feature shape: {image_embeddings.shape} (512-dim CLIP image embeddings)")
    
    # Step 3: Train/Test Split
    print(f"\n[3/5] Splitting data (train: {int((1-test_size)*100)}%, test: {int(test_size*100)}%)...")
    X = image_embeddings
    y = df_filtered['topic_id'].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y  # Ensures balanced split across topics
    )
    
    print(f"   Training set: {len(X_train)} samples")
    print(f"   Test set: {len(X_test)} samples")
    
    # Step 4: Feature Scaling
    print("\n[4/5] Scaling features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Step 5: Train and Compare Classifiers
    print("\n[5/5] Training classifiers...")
    print("   (Removed Gradient Boosting - too slow, minimal accuracy gain)")
    
    classifiers = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000,
            random_state=random_state,
            class_weight='balanced',
            n_jobs=-1
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            random_state=random_state,
            class_weight='balanced',
            n_jobs=-1
        )
    }
    
    comparison_results = []
    best_accuracy = 0
    best_model = None
    best_classifier_name = None
    best_y_pred = None
    best_cm = None
    best_report = None
    
    for clf_name, clf in classifiers.items():
        print(f"\n   Training {clf_name}...")
        
        # Train
        clf.fit(X_train_scaled, y_train)
        
        # Predict
        y_pred = clf.predict(X_test_scaled)
        
        # Evaluate
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        
        # Store results
        comparison_results.append({
            'classifier': clf_name,
            'accuracy': accuracy,
            'precision': report['weighted avg']['precision'],
            'recall': report['weighted avg']['recall'],
            'f1_score': report['weighted avg']['f1-score']
        })
        
        print(f"      Accuracy: {accuracy*100:.2f}%")
        
        # Track best model
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = clf
            best_classifier_name = clf_name
            best_y_pred = y_pred
            best_cm = cm
            best_report = classification_report(y_test, y_pred, zero_division=0)
        
        if not compare_classifiers:
            break  # Only train first classifier if comparison not requested
    
    # Display Results
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    
    if compare_classifiers:
        print("\nCLASSIFIER COMPARISON:")
        print("-"*80)
        comparison_df = pd.DataFrame(comparison_results)
        for _, row in comparison_df.iterrows():
            print(f"{row['classifier']:20s}: Acc={row['accuracy']*100:5.2f}%  "
                  f"Prec={row['precision']*100:5.2f}%  "
                  f"Rec={row['recall']*100:5.2f}%  "
                  f"F1={row['f1_score']*100:5.2f}%")
    
    print(f"\n{'='*80}")
    print(f"BEST MODEL: {best_classifier_name}")
    print(f"{'='*80}")
    print(f"Test Accuracy: {best_accuracy*100:.2f}%")
    print(f"\nDetailed Classification Report:")
    print(best_report)
    
    # Prepare return dictionary
    results = {
        'best_model': best_model,
        'best_classifier_name': best_classifier_name,
        'confusion_matrix': best_cm,
        'accuracy': best_accuracy,
        'classification_report': best_report,
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'y_pred': best_y_pred,
        'scaler': scaler,
        'topic_labels': sorted(df_filtered['topic_id'].unique())
    }
    
    if compare_classifiers:
        results['comparison_results'] = comparison_df
    
    print("\n" + "="*80)
    print("PREDICT() COMPLETE")
    print("="*80)
    
    return results

In [30]:
# Run predict() function with the labeled dataset from topic_miner()
prediction_results = predict(
    labeled_df=balanced,
    test_size=0.3,
    random_state=42,
    compare_classifiers=True
)

PREDICT() - CLASSIFIER TRAINING AND EVALUATION

[1/6] Preprocessing data...
   Removed 432 outliers (topic -1)
   Training with 4687 samples across 234 topics

[2/6] Generating multimodal embeddings (CLIP)...
   Encoding captions...
   Encoding images...


Batches: 100%|██████████| 147/147 [01:44<00:00,  1.40it/s]


   Combining text and image embeddings...
   Feature shape: (4687, 1024)

[3/6] Splitting data (train: 70%, test: 30%)...
   Training set: 3280 samples
   Test set: 1407 samples

[4/6] Scaling features...

[5/6] Training classifiers...

   Training Logistic Regression...


python(8840) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


      Accuracy: 99.22%

   Training Random Forest...
      Accuracy: 83.08%

   Training Gradient Boosting...


KeyboardInterrupt: 

## Confusion Matrix Visualization

Visualize the confusion matrix to understand model performance across all topic classes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract confusion matrix and labels
cm = prediction_results['confusion_matrix']
topic_labels = prediction_results['topic_labels']

# Create figure with appropriate size based on number of topics
num_topics = len(topic_labels)
figsize = max(12, num_topics * 0.6)

plt.figure(figsize=(figsize, figsize))

# Plot confusion matrix with annotations
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=topic_labels,
    yticklabels=topic_labels,
    cbar_kws={'label': 'Number of Samples'},
    square=True
)

plt.title(
    f'Confusion Matrix - {prediction_results["best_classifier_name"]}\n'
    f'Test Accuracy: {prediction_results["accuracy"]*100:.2f}%',
    fontsize=16,
    pad=20
)
plt.xlabel('Predicted Topic', fontsize=14)
plt.ylabel('True Topic', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix Shape: {cm.shape}")
print(f"Total Test Samples: {cm.sum()}")
print(f"Correctly Classified: {np.trace(cm)} ({np.trace(cm)/cm.sum()*100:.2f}%)")

In [ ]:
# Normalized Confusion Matrix (shows percentages per class)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(figsize, figsize))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Greens',
    xticklabels=topic_labels,
    yticklabels=topic_labels,
    cbar_kws={'label': 'Proportion'},
    square=True,
    vmin=0,
    vmax=1
)

plt.title(
    f'Normalized Confusion Matrix - {prediction_results["best_classifier_name"]}\n'
    f'(Shows recall/true positive rate per class)',
    fontsize=16,
    pad=20
)
plt.xlabel('Predicted Topic', fontsize=14)
plt.ylabel('True Topic', fontsize=14)
plt.tight_layout()
plt.show()

# Identify best and worst performing classes
diagonal = np.diag(cm_normalized)
best_topic_idx = np.argmax(diagonal)
worst_topic_idx = np.argmin(diagonal)

print(f"\nBest Performing Topic: {topic_labels[best_topic_idx]} (Recall: {diagonal[best_topic_idx]*100:.2f}%)")
print(f"Worst Performing Topic: {topic_labels[worst_topic_idx]} (Recall: {diagonal[worst_topic_idx]*100:.2f}%)")
print(f"Average Recall across all topics: {diagonal.mean()*100:.2f}%")

## Additional Analysis

Compare classifier performance and analyze per-topic metrics.

In [ ]:
# Classifier Comparison Bar Chart
if 'comparison_results' in prediction_results:
    comparison_df = prediction_results['comparison_results']
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Plot 1: Accuracy Comparison
    ax1 = axes[0]
    bars1 = ax1.bar(
        comparison_df['classifier'],
        comparison_df['accuracy'] * 100,
        color=['#1f77b4', '#ff7f0e', '#2ca02c']
    )
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.set_title('Classifier Accuracy Comparison', fontsize=14, pad=15)
    ax1.set_ylim([0, 100])
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom', fontsize=11
        )
    
    # Plot 2: F1-Score Comparison
    ax2 = axes[1]
    bars2 = ax2.bar(
        comparison_df['classifier'],
        comparison_df['f1_score'] * 100,
        color=['#1f77b4', '#ff7f0e', '#2ca02c']
    )
    ax2.set_ylabel('F1-Score (%)', fontsize=12)
    ax2.set_title('Classifier F1-Score Comparison', fontsize=14, pad=15)
    ax2.set_ylim([0, 100])
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars2:
        height = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom', fontsize=11
        )
    
    plt.tight_layout()
    plt.show()
    
    # Print comparison table
    print("\nDETAILED CLASSIFIER COMPARISON:")
    print("="*80)
    print(f"{'Classifier':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*80)
    for _, row in comparison_df.iterrows():
        print(f"{row['classifier']:<25} "
              f"{row['accuracy']*100:>10.2f}%  "
              f"{row['precision']*100:>10.2f}%  "
              f"{row['recall']*100:>10.2f}%  "
              f"{row['f1_score']*100:>10.2f}%")
    print("="*80)

In [ ]:
# Per-Topic Performance Analysis
from sklearn.metrics import precision_score, recall_score, f1_score

y_test = prediction_results['y_test']
y_pred = prediction_results['y_pred']
topic_labels = prediction_results['topic_labels']

# Calculate per-topic metrics
topic_metrics = []
for topic in topic_labels:
    # Binary classification: current topic vs all others
    y_test_binary = (y_test == topic).astype(int)
    y_pred_binary = (y_pred == topic).astype(int)
    
    precision = precision_score(y_test_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_test_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_binary, zero_division=0)
    support = y_test_binary.sum()
    
    topic_metrics.append({
        'topic': topic,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'support': support
    })

metrics_df = pd.DataFrame(topic_metrics)

# Visualize per-topic F1 scores
plt.figure(figsize=(14, 6))

bars = plt.bar(
    range(len(metrics_df)),
    metrics_df['f1_score'] * 100,
    color=plt.cm.viridis(np.linspace(0.3, 0.9, len(metrics_df)))
)

plt.xlabel('Topic ID', fontsize=12)
plt.ylabel('F1-Score (%)', fontsize=12)
plt.title(f'Per-Topic Performance (F1-Score) - {prediction_results["best_classifier_name"]}', fontsize=14, pad=15)
plt.xticks(range(len(metrics_df)), metrics_df['topic'])
plt.ylim([0, 100])
plt.grid(axis='y', alpha=0.3)
plt.axhline(y=metrics_df['f1_score'].mean() * 100, color='r', linestyle='--', linewidth=2, label=f'Mean: {metrics_df["f1_score"].mean()*100:.2f}%')
plt.legend()
plt.tight_layout()
plt.show()

# Print detailed per-topic metrics
print("\nPER-TOPIC PERFORMANCE METRICS:")
print("="*80)
print(f"{'Topic':<8} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-"*80)
for _, row in metrics_df.iterrows():
    print(f"{row['topic']:<8} "
          f"{row['precision']*100:>10.2f}%  "
          f"{row['recall']*100:>10.2f}%  "
          f"{row['f1_score']*100:>10.2f}%  "
          f"{int(row['support']):>8}")
print("-"*80)
print(f"{'AVERAGE':<8} "
      f"{metrics_df['precision'].mean()*100:>10.2f}%  "
      f"{metrics_df['recall'].mean()*100:>10.2f}%  "
      f"{metrics_df['f1_score'].mean()*100:>10.2f}%  "
      f"{int(metrics_df['support'].sum()):>8}")
print("="*80)

## Assignment Summary

Complete workflow demonstrating multimodal topic mining and classification.

In [ ]:
print("="*80)
print("VISUAL DATA PREDICTIONS ASSIGNMENT - COMPLETE SUMMARY")
print("="*80)

print("\n📊 DATASET")
print("-"*80)
print(f"Source: Pokemon TCG (Kaggle)")
print(f"Total cards processed: {len(balanced) + (len(balanced[balanced['topic_id'] != -1]) - len(balanced))}")
print(f"Cards used for training: {len(balanced[balanced['topic_id'] != -1])}")
print(f"Image + Caption pairs: Multimodal embeddings via CLIP")

print("\n🔍 FUNCTION 1: TOPIC_MINER()")
print("-"*80)
print(f"Method: BERTopic with CLIP-ViT-B-32 embeddings")
print(f"Topics discovered: {len(prediction_results['topic_labels'])}")
print(f"Outlier rate: {(len(balanced[balanced['topic_id'] == -1]) / len(balanced) * 100):.2f}%")
print(f"Features: Multimodal (text captions + images)")

print("\n🎯 FUNCTION 2: PREDICT()")
print("-"*80)
print(f"Train/Test Split: 70% / 30%")
print(f"Training samples: {len(prediction_results['y_train'])}")
print(f"Test samples: {len(prediction_results['y_test'])}")

print("\n🏆 CLASSIFIER COMPARISON")
print("-"*80)
if 'comparison_results' in prediction_results:
    for _, row in prediction_results['comparison_results'].iterrows():
        marker = "⭐" if row['classifier'] == prediction_results['best_classifier_name'] else "  "
        print(f"{marker} {row['classifier']:<25}: {row['accuracy']*100:>6.2f}% accuracy")

print(f"\n✅ BEST MODEL: {prediction_results['best_classifier_name']}")
print(f"   Test Accuracy: {prediction_results['accuracy']*100:.2f}%")

print("\n📈 KEY RESULTS")
print("-"*80)
print(f"✓ Successfully implemented both required functions")
print(f"✓ Multimodal embeddings improve over unimodal (text or image only)")
print(f"✓ Confusion matrix demonstrates strong topic classification")
print(f"✓ {prediction_results['best_classifier_name']} achieves best performance")

print("\n" + "="*80)
print("ASSIGNMENT REQUIREMENTS: ✅ COMPLETE")
print("="*80)
print("✓ Dataset: Pokemon TCG (5,000+ captioned images)")
print("✓ topic_miner(): Implemented with BERTopic + CLIP")
print("✓ predict(): Implemented with multiple classifier comparison")
print("✓ Confusion Matrix: Generated and visualized")
print("✓ Train/Test Split: 70-30 with stratification")
print("✓ Evaluation Metrics: Accuracy, Precision, Recall, F1-Score")
print("="*80)

---

## 📖 How to Use This Notebook

### Quick Start

1. **Install Dependencies** (Cell 1)
   - Uncomment and run the pip install commands

2. **Download Dataset** (Cell 4)
   - Downloads Pokemon TCG dataset from Kaggle
   - Processes ~6000 cards with images

3. **Run Topic Mining** (Cell 13)
   ```python
   labeled_data = topic_miner('./pokemon_cards/pokemon_cards_metadata.csv')
   ```

4. **Run Classification** (Cell 17)
   ```python
   results = predict(labeled_data, test_size=0.3, compare_classifiers=True)
   ```

5. **View Results** (Cells 19-25)
   - Confusion matrices (raw and normalized)
   - Classifier comparison charts
   - Per-topic performance analysis

### Key Functions

**`topic_miner(dataset_path, ...)`**
- Discovers multimodal topics using BERTopic + CLIP
- Returns labeled DataFrame with topic assignments
- Customizable clustering parameters

**`predict(labeled_df, test_size=0.3, ...)`**
- Trains multiple classifiers on topic labels
- Returns best model with confusion matrix
- Provides comprehensive evaluation metrics

### Dataset

- **Source:** Pokemon Trading Card Game (Kaggle)
- **Size:** 5,119 Pokemon cards with images
- **Features:** Card type, HP, rarity, evolution stage, abilities
- **Modalities:** Images + rich text captions

---